# Do computed tools beat LLM math? A Phoenix experiment on my Garmin data
**Question:** Does moving computation into tools improve numeric accuracy for an agent answering questions about my running history?

## 1. Setup and tracing

In [1]:
from phoenix.otel import register
register(project_name='garmin-run-agent', auto_instrument=True)
from data import load_runs
runs = load_runs(); runs.tail()

/Users/treviemeister/Downloads/garmin-agent/.venv-1/lib/python3.12/site-packages/phoenix/otel/otel.py:449: UserWarning: Could not infer collector endpoint protocol, defaulting to HTTP.
  warnings.warn("Could not infer collector endpoint protocol, defaulting to HTTP.")


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: garmin-run-agent
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: https://app.phoenix.arize.com/s/trevorsantiago25/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {'authorization': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



,date,distance_mi,duration_s,avg_hr,ascent_ft,pace_min_per_mi
156,2026-09-23 08:59:12,6.21,2949.0,152,737.0,7.914654
157,2026-09-24 09:05:36,1.76,902.0,136,61.0,8.541667
158,2026-09-24 09:22:53,0.78,377.9,148,30.0,8.074786
159,2026-09-25 10:39:55,5.03,2273.0,153,211.0,7.531478
160,2026-09-28 09:06:21,5.04,2356.0,145,187.0,7.791005


## 2. Ground truth (pandas, independent of the agent)

In [ ]:
from ground_truth import build_questions
import pandas as pd
qs = pd.DataFrame(build_questions()); qs[['question','expected']]

## 3. One traced run per design (open Phoenix and compare the traces)

In [ ]:
from agent import run_agent
q = 'How many miles did I run last week? (answer in miles)'
print('A:', run_agent(q, 'A_raw'))
print('B:', run_agent(q, 'B_computed'))

## 4. Experiments
Run `python run_experiments.py`, or call `main()` here.

In [ ]:
from run_experiments import main
# main()  # already run (see Phoenix). Re-running creates NEW experiments and costs API credits.


## 5. What the traces showed

**Setup:** 

Claude Haiku 4.5 with the same prompt and 22 questions, on my own Garmin running data. (161 runs, Aug 2025 – Sep 2026).

Each design ran once per question as a Phoenix experiment on dataset `garmin-run-questions-v1`.

Answers are scored by a code evaluator against pandas ground truth.

For each wrong answer, I read the trace and checked two things separately:
1. **Did the model pick the right rows?** (date logic)
2. **Do the numbers it listed add up to what it claimed?** (arithmetic)

| Q | Question | Design | Got | Correct | Failure mode | Right rows? |
|---|---|---|---|---|---|---|
| 4 | August miles | A | 96.83 | 77.53 | Arithmetic: summing 21 values | ✅ |
| 5 | August pace | A | 7.22 | 8.19 | Arithmetic: both distance and time totals wrong | ✅ |
| 6 | September pace | A | 8.62 | 8.15 | Arithmetic: time total 650 min vs actual 582 | ✅ |
| 12 | August ascent | A | 5354 | 4353 | Arithmetic: off by exactly 1,001 | ✅ |
| 13 | Sep avg distance | A | 3.65 | 2.86 | Arithmetic: summing 25 values | ✅ |
| 15 | Sep − Aug miles | A | −20.39 | −6.10 | Arithmetic: both monthly totals wrong | ✅ |
| 17 | Sep avg HR | A | 144 | 146.1 | Arithmetic: sum 3608 vs actual 3652 | ✅ |
| 19 | Year-to-date miles | A | 523.29 | 436.16 | Arithmetic: summing 134 values | ✅ |
| 18 | Runs/week, last 8 weeks | A | 5.0 | 5.62 | Counting: tallied 40 runs instead of 45; window off by one day | ~ |
| 18 | Runs/week, last 8 weeks | B | 3.0 | 5.62 | Tool-coverage gap: no tool resolves "last 8 full weeks", so the model guessed Sep 1–27 and still divided by 8 | ❌ |

**Traces worth opening** (Phoenix → Datasets → `garmin-run-questions-v1` → Experiments → compare, then click a row):
- **Q4, A_raw:** the clearest mental-math failure. The rows are right and the total is wrong.
- **Q13 vs Q15, A_raw:** the same 25 September distances, summed differently in each run.

## 6. Results

| Design | Accuracy | Tool calls (total / per question) | Avg latency |
|---|---|---|---|
| **A_raw**: LLM does date logic + math | **13/22 (59%)** | 25 / 1.14 | 3.2 s |
| **B_computed**: tools do date logic + math | **21/22 (95%)** | 30 / 1.36 | 2.5 s |

- **All 8 of A's arithmetic failures used exactly the right rows.** A never mis-scoped a calendar month or "last week". The only date slip (Q18, one day off) didn't change which runs were counted.
- **The errors aren't reproducible.** The model wrote out the same 21 August distances three times, in Q4, Q5 and Q15, and got 96.83, 85.73 and 93.83. The true total is 77.53. The same happened with September: 75.43, 91.24 and 73.44 (true 71.43).
- **B uses more tool calls but finishes faster.** B spends an extra call on `resolve_date_range`, but A writes out long arithmetic, which makes it slower overall.
- **Aerobic efficiency (v2, 6 extra questions):** _pending. It uses a new dataset, `garmin-run-questions-v2`, so v1 stays unchanged._

## 7. Takeaways for developers
1. **Test "what the model does" and "what the model knows" separately.** My hypothesis was that date logic and arithmetic would both fail. The traces showed that date resolution was fine (8/8 correct row sets) and arithmetic was the entire problem. A single accuracy number can't show that distinction; reading the trace for each failure can.
2. **LLM arithmetic over many values fails, and fails differently each run.** Summing 20+ numbers in context got 0 of 8 right, with a different wrong total each time for the same input. Any numeric answer derived from more than a handful of values should come from a tool (or code execution), not from generated text.
3. **Computed tools only help inside their coverage.** B's one miss was a question its date tool couldn't express ("last 8 full weeks"). The model didn't report the gap; it quietly guessed. Tool-based designs need evals that probe the edges of the tool set, not just the common paths.

## 8. What went wrong?  Next Steps.
- **Limitations:** one run per question (n=22 per design), so A's accuracy would vary on a rerun. `repetitions=3` would fix that. One model only. Heart-rate-based metrics are noisy (heat, hills, sleep), and I did no workout vs easy-run filtering.
- **Cut for time:** an LLM-judge eval, a second model, and repeated runs.
- **Data cleaning:** I dropped one 0.00-mile activity and kept ~7 accidental sub-0.5-mile starts. Both designs and the ground truth see the same rows, so the comparison stays fair.
Full list of problems and fixes: `CHALLENGES.md`.
